# 10 - Unit Economics & Hypotheses

**Goal:** compute unit economics by product, identify growth points via sensitivity analysis, form hypotheses, and design A/B experiments to test them.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_PROCESSED = Path('../data/processed')

deals = pd.read_parquet(DATA_PROCESSED / 'deals_clean.parquet')
spend = pd.read_parquet(DATA_PROCESSED / 'spend_clean.parquet')

## 1. Unit economics by product

In [3]:
MAIN_PRODUCTS = ['Web Developer', 'Digital Marketing', 'UX/UI Design']

In [4]:
# Check
print(f"Deals: {len(deals)}")
print(f"Spend records: {len(spend)}")
print(f"Unique contacts: {deals['Contact Name'].nunique()}")
print(f"Won Confirmed: {deals['is_won_confirmed'].sum()}")

Deals: 21593
Spend records: 19862
Unique contacts: 18089
Won Confirmed: 839


In [5]:
def get_UA(deals):
    """UA — unique contacts in the funnel. Shared pool, same for all products."""
    return deals['Contact Name'].nunique()


def get_CPA(spend, deals):
    """CPA — cost per contact in the funnel. Shared across products."""
    total_spend = spend['Spend'].sum()
    ua = get_UA(deals)
    return total_spend / ua


def get_B(product, deals):
    """B — number of buyers per product (is_won_confirmed)."""
    mask = (deals['Product'] == product) & deals['is_won_confirmed']
    return int(mask.sum())


def get_C1(product, deals):
    """C1 — conversion to buyers. C1 = B / UA."""
    return get_B(product, deals) / get_UA(deals)


def get_T(product, deals):
    """T — total transactions per product = sum of Months of study for Won."""
    mask = (deals['Product'] == product) & deals['is_won_confirmed']
    return int(deals.loc[mask, 'Months of study'].sum())


def get_APC(product, deals):
    """APC — average transactions per buyer. APC = T / B."""
    b = get_B(product, deals)
    if b == 0:
        return 0
    return get_T(product, deals) / b


def get_Revenue(product, deals):
    """Revenue — total revenue per product = sum(revenue_actual) for Won."""
    mask = (deals['Product'] == product) & deals['is_won_confirmed']
    return float(deals.loc[mask, 'revenue_actual'].sum())


def get_AOV(product, deals):
    """AOV — average transaction value. AOV = Revenue / T."""
    t = get_T(product, deals)
    if t == 0:
        return 0
    return get_Revenue(product, deals) / t


def get_CLTV(product, deals):
    """CLTV — revenue per buyer. CLTV = AOV x APC."""
    return get_AOV(product, deals) * get_APC(product, deals)


def get_LTV(product, deals):
    """LTV — revenue per acquired contact. LTV = CLTV x C1."""
    return get_CLTV(product, deals) * get_C1(product, deals)


def get_margin_profit(product, deals, spend):
    """Margin profit = (LTV - CPA) x UA."""
    ltv = get_LTV(product, deals)
    cpa = get_CPA(spend, deals)
    ua = get_UA(deals)
    return (ltv - cpa) * ua

In [6]:
def build_unit_economics_table(products, deals, spend):
    """Builds the unit-economics table by product."""
    rows = []
    for product in products:
        row = {
            'Product': product,
            'UA': get_UA(deals),
            'CPA': round(get_CPA(spend, deals), 2),
            'B': get_B(product, deals),
            'C1, %': round(get_C1(product, deals) * 100, 2),
            'T': get_T(product, deals),
            'APC': round(get_APC(product, deals), 2),
            'AOV': round(get_AOV(product, deals), 2),
            'Revenue': round(get_Revenue(product, deals), 0),
            'CLTV': round(get_CLTV(product, deals), 2),
            'LTV': round(get_LTV(product, deals), 2),
            'Margin profit': round(get_margin_profit(product, deals, spend), 0),
        }
        rows.append(row)
    
    df = pd.DataFrame(rows).set_index('Product')
    return df

ue_table = build_unit_economics_table(MAIN_PRODUCTS, deals, spend)
print(ue_table.T)  # transpose for readability — products in columns

Product        Web Developer  Digital Marketing  UX/UI Design
UA                  18089.00           18089.00      18089.00
CPA                     8.27               8.27          8.27
B                     137.00             474.00        228.00
C1, %                   0.76               2.62          1.26
T                     505.00            2897.00       1170.00
APC                     3.69               6.11          5.13
AOV                   725.90             777.12        811.15
Revenue            366580.00         2251320.00     949045.00
CLTV                 2675.77            4749.62       4162.48
LTV                    20.27             124.46         52.47
Margin profit      217057.00         2101797.00     799522.00


### Unit-economics summary

**Positive unit economics for all three products.** LTV > CPA for each:
- DM: LTV/CPA = 14.9 (€123.60 / €8.27)
- UX/UI: LTV/CPA = 6.3
- WD: LTV/CPA = 2.6

LTV/CPA >= 3 is considered healthy. Web Developer is borderline.

**Main driver — Digital Marketing.** Margin profit €2.09M of €3.11M school-wide (67%). High C1 (2.62%) at a moderate AOV.

**Web Developer — the weak link.** Low C1 (0.76%) and a short course give the lowest LTV. Still profitable.

**AOV is almost identical** across products (€765-801). The LTV difference is driven by **C1** (how many were acquired) and **APC** (how many times each paid).

## 2. Growth points from unit economics (sensitivity)

In [7]:
def margin_profit_from_metrics(UA, C1, AOV, APC, CPA):
    """
    Computes margin profit from metric values.
    Used for sensitivity analysis (what if we improve a metric?).
    """
    CLTV = AOV * APC
    LTV = CLTV * C1
    return (LTV - CPA) * UA

In [8]:
# Control check on Digital Marketing
UA = get_UA(deals)
CPA = get_CPA(spend, deals)
C1 = get_C1('Digital Marketing', deals)
AOV = get_AOV('Digital Marketing', deals)
APC = get_APC('Digital Marketing', deals)

margin_check = margin_profit_from_metrics(UA, C1, AOV, APC, CPA)
print(f"Control:    €{margin_check:,.0f}")
print(f"Should be:  €{get_margin_profit('Digital Marketing', deals, spend):,.0f}")

Control:    €2,101,797
Should be:  €2,101,797


In [9]:
def sensitivity_analysis(product, deals, spend, delta=0.10):
    """
    For each metric (UA, C1, AOV, APC, CPA):
    - Improve it by delta (decrease for CPA, increase for the rest)
    - Compute the new margin profit
    - Return a table: metric, baseline margin, new margin, uplift
    """
    UA_base = get_UA(deals)
    CPA_base = get_CPA(spend, deals)
    C1_base = get_C1(product, deals)
    AOV_base = get_AOV(product, deals)
    APC_base = get_APC(product, deals)
    
    margin_base = margin_profit_from_metrics(
        UA_base, C1_base, AOV_base, APC_base, CPA_base
    )
    
    scenarios = {
        'UA  (+10%)':  (UA_base * (1 + delta), C1_base, AOV_base, APC_base, CPA_base),
        'C1  (+10%)':  (UA_base, C1_base * (1 + delta), AOV_base, APC_base, CPA_base),
        'AOV (+10%)':  (UA_base, C1_base, AOV_base * (1 + delta), APC_base, CPA_base),
        'APC (+10%)':  (UA_base, C1_base, AOV_base, APC_base * (1 + delta), CPA_base),
        'CPA (-10%)':  (UA_base, C1_base, AOV_base, APC_base, CPA_base * (1 - delta)),
    }
    
    rows = []
    for name, params in scenarios.items():
        new_margin = margin_profit_from_metrics(*params)
        delta_margin = new_margin - margin_base
        delta_pct = delta_margin / margin_base * 100
        rows.append({
            'Scenario': name,
            'Margin (new)': round(new_margin, 0),
            'Δ Margin, €': round(delta_margin, 0),
            'Δ Margin, %': round(delta_pct, 2),
        })
    
    df = pd.DataFrame(rows)
    df = df.sort_values('Δ Margin, €', ascending=False)
    return df, margin_base


# Test on Digital Marketing
sens_dm, base_dm = sensitivity_analysis('Digital Marketing', deals, spend)
print(f"Baseline margin for Digital Marketing: €{base_dm:,.0f}\n")
print(sens_dm.to_string(index=False))

Baseline margin for Digital Marketing: €2,101,797

  Scenario  Margin (new)  Δ Margin, €  Δ Margin, %
C1  (+10%)     2326929.0     225132.0        10.71
APC (+10%)     2326929.0     225132.0        10.71
AOV (+10%)     2326929.0     225132.0        10.71
UA  (+10%)     2311976.0     210180.0        10.00
CPA (-10%)     2116749.0      14952.0         0.71


In [10]:
def all_products_sensitivity(products, deals, spend, delta=0.10):
    """Combines sensitivity for all products into one table."""
    all_rows = []
    for product in products:
        sens_df, base = sensitivity_analysis(product, deals, spend, delta)
        for _, row in sens_df.iterrows():
            all_rows.append({
                'Product': product,
                'Scenario': row['Scenario'],
                'Margin (new)': row['Margin (new)'],
                'Δ Margin, €': row['Δ Margin, €'],
                'Δ Margin, %': row['Δ Margin, %'],
            })
    return pd.DataFrame(all_rows)


all_sens = all_products_sensitivity(MAIN_PRODUCTS, deals, spend)
print(all_sens.to_string(index=False))

          Product   Scenario  Margin (new)  Δ Margin, €  Δ Margin, %
    Web Developer C1  (+10%)      253715.0      36658.0        16.89
    Web Developer APC (+10%)      253715.0      36658.0        16.89
    Web Developer AOV (+10%)      253715.0      36658.0        16.89
    Web Developer UA  (+10%)      238762.0      21706.0        10.00
    Web Developer CPA (-10%)      232009.0      14952.0         6.89
Digital Marketing C1  (+10%)     2326929.0     225132.0        10.71
Digital Marketing APC (+10%)     2326929.0     225132.0        10.71
Digital Marketing AOV (+10%)     2326929.0     225132.0        10.71
Digital Marketing UA  (+10%)     2311976.0     210180.0        10.00
Digital Marketing CPA (-10%)     2116749.0      14952.0         0.71
     UX/UI Design C1  (+10%)      894426.0      94904.0        11.87
     UX/UI Design APC (+10%)      894426.0      94904.0        11.87
     UX/UI Design AOV (+10%)      894426.0      94904.0        11.87
     UX/UI Design UA  (+10%)      

### Findings

For all three products, the best growth points are C1, APC, and AOV — improving any of them yields the same uplift in margin profit. Notably, for Web Developer a CPA decrease has a visible effect, while for the others it doesn't.

For hypothesis testing, we take C1 as the growth point.

## 3. Map hypotheses to product metrics

### Hypotheses

- **Digital Marketing:** reducing SLA to <1h raises C1 by 1%.
- **UX/UI Design:** a free trial lesson raises C1 by 2%.
- **Web Developer:** rebalancing budget toward effective channels (e.g. YouTube, Instagram) raises C1 by 3%.

## 4. A/B-experiment parameters for the C1 hypotheses

In [11]:
def calculate_experiment(product, deals, mde_abs):
    """
    Compute A/B-experiment parameters for a C1 hypothesis.

    Params:
    mde_abs: MDE in absolute percentage points (0.01 = 1%)

    Returns: sample size and experiment duration in days.
    """
    C1 = get_C1(product, deals)
    
    # Sample size per group
    n = (16 * C1 * (1 - C1)) / (mde_abs ** 2)
    
    # Total size (control + test)
    n_total = 2 * n
    
    # Product DAU = product leads / days in data
    product_leads = (deals['Product'] == product).sum()
    days_in_data = (deals['Created Time'].max() - deals['Created Time'].min()).days
    DAU = product_leads / days_in_data
    
    duration_days = n_total / DAU
    
    return {
        'Product': product,
        'C1 current, %': round(C1 * 100, 2),
        'MDE, %': round(mde_abs * 100, 2),
        'C1 target, %': round((C1 + mde_abs) * 100, 2),
        'n (per group)': round(n, 0),
        'n_total': round(n_total, 0),
        'Product leads (total)': product_leads,
        'Days in data': days_in_data,
        'DAU': round(DAU, 2),
        'Duration, days': round(duration_days, 0),
    }


# Calculation for three products with different MDE
experiments_config = [
    ('Digital Marketing', 0.01),   # MDE 1% abs.
    ('UX/UI Design',      0.02),   # MDE 2% abs.
    ('Web Developer',     0.03),   # MDE 3% abs.
]

experiment_results = pd.DataFrame([
    calculate_experiment(product, deals, mde)
    for product, mde in experiments_config
])

print(experiment_results.T)

                                       0             1              2
Product                Digital Marketing  UX/UI Design  Web Developer
C1 current, %                       2.62          1.26           0.76
MDE, %                               1.0           2.0            3.0
C1 target, %                        3.62          3.26           3.76
n (per group)                     4083.0         498.0          134.0
n_total                           8165.0         996.0          267.0
Product leads (total)               1990          1022            575
Days in data                         353           353            353
DAU                                 5.64           2.9           1.63
Duration, days                    1448.0         344.0          164.0


### Findings

**Main constraint:** at the chosen MDEs, a classic A/B test on C1 requires from 5 months to 4 years. This is consistent with the real conditions: low base conversion (0.76-2.62%) and limited lead flow (1.6-5.6 per day per product) don't allow quickly reaching a statistically significant result for small changes.

## Hypothesis-testing method: A/B experiment

### Approach

For each hypothesis, run a parallel A/B experiment: leads are randomly split into a control group (old process) and a test group (with the change). After the calculated number of days, compare the C1 of the two groups.

### Metric

**Primary:** C1 = Won / UA
**Guardrail:** lead volume, AOV, time-to-close — should not drop significantly in the test group.

### Sample-size formula

n = (16 × C1 × (1-C1)) / MDE²

where:
- 16 — coefficient for significance level alpha=0.05 and power 80%
- C1 — current conversion (as a fraction)
- MDE — minimum detectable effect (as a fraction)

Total experiment size = 2n (n per group).

### Duration

Duration = 2n / DAU, where DAU = (product leads) / (days in data).

### Validity conditions

1. **Randomization** — leads assigned to A and B at random.
2. **Parallelism** — both groups run simultaneously.
3. **Change isolation** — only the tested variable differs between groups.
4. **Segment homogeneity** — test within a single product.
5. **Sufficient duration** — don't stop before reaching the calculated n.
6. **External-factor control** — pause the experiment on significant external events.

### Result criteria

**Confirmed:** C1 difference >= MDE, p < 0.05, guardrail metrics stable -> scale to 100% of traffic.

**Not confirmed:** effect below MDE or not significant, or guardrails dropped -> roll back, look for another hypothesis.

### Limitations

- For Digital Marketing the calculated duration is 1,448 days (~4 years) — not applicable in a real business. Either raise MDE to 2-3% (test larger changes) or use alternative methods (quasi-experiment, before/after with seasonality control).
- An A/B test checks one hypothesis at a time — changes can't be combined in one experiment.
- Results apply only to the tested segment and product.
- The method is sensitive to validity violations — any breach makes the result unreliable.

### Recommended order

Start with **Web Developer** as the fastest (~164 days). This allows:
1. Validating the methodology on a compact test.
2. Getting first results within the school's study cycle.
3. Calibrating expectations before the longer UX/UI and DM experiments.